# Advertisement Success Dataset

## Task
This is a binary classification problem where you need to predict whether an ad will lead to a netgain

Loading the Nesscary libraries required

In [ ]:
from sklearn.metrics import make_scorer, accuracy_score,auc
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import ExtraTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn import preprocessing
from scipy.stats import norm
import matplotlib.pylab as pylab
import matplotlib.pyplot as plt
from pandas import get_dummies
import matplotlib as mpl
from scipy import stats
import xgboost as xgb
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib
import warnings
import sklearn
import scipy
import numpy
import json
import sys
import csv


In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
warnings.filterwarnings('ignore')
%matplotlib inline

# Loading the train and test data-set using pandas.read_csv



In [ ]:
df_train=pd.read_csv('/kaggle/input/advertsuccess/Train.csv')
df_test=pd.read_csv('/kaggle/input/advertsuccess/Test.csv')

# Data Description:
Train.csv : 26049 x 12 [including headers] : training data set

Test.csv : 6514 x 11 [including headers] : test data set

# Data

Data Description

id                                 -Unique id for each row

ratings                            -Metric out of 1 which represents how much of the targeted demographic watched the advertisement

airlocation                        -Country of origin

airtime                            -Time when the advertisement was aired

average_runtime(minutes_per_week)  -Minutes per week the advertisement was aired

targeted_sex                       -Sex that was mainly targeted for the advertisement

genre                              -The type of advertisement

industry                           -The industry to which the product belonged

economic_status                    -The economic health during which the show aired

relationship_status                -The relationship status of the most responsive customers to the advertisement

expensive                          -A general measure of how expensive the product or service is that the ad is discussing.

money_back_guarantee               -Whether or not the product offers a refund in the case of customer dissatisfaction.

netgain [target]                   -Whether the ad will incur a gain or loss when sold

In [ ]:
df_train.head()

In [ ]:
#function for missing data
def missing_data(df_train):
    total = df_train.isnull().sum().sort_values(ascending=False)
    percent = (df_train.isnull().sum()/df_train.isnull().count()).sort_values(ascending=False)
    missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
    return(missing_data.head(20))

In [ ]:
missing_data(df_train)

In [ ]:
df_train.describe()

In [ ]:
df_train.dtypes

In [ ]:
df_train['netgain']=df_train['netgain'].astype('str')

In [ ]:
df_train.netgain[df_train.netgain == 'True'] = 1
df_train.netgain[df_train.netgain == 'False'] = 0

In [ ]:
df_train['netgain']=df_train['netgain'].astype('int')

In [ ]:
df_train=df_train[df_train.airlocation != 'Holand-Netherlands']

In [ ]:
df_train.airlocation.value_counts()

# Correlation
When two sets of data are strongly linked together we say they have a High Correlation.

The word Correlation is made of Co- (meaning "together"), and Relation

Correlation is Positive when the values increase together, and
Correlation is Negative when one value decreases as the other increases
A correlation is assumed to be linear (following a line).

correlation examples
Correlation can have a value:

1 is a perfect positive correlation
0 is no correlation (the values don't seem linked at all)
-1 is a perfect negative correlation
The value shows how good the correlation is (not how steep the line is), and if it is positive or negative.

In [ ]:
#correlation matrix
corrmat = df_train.corr()
f, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corrmat, vmax=.8, square=True);

# Data Visualization

Data visualization is the graphic representation of data. It involves producing images that communicate relationships among the represented data to viewers of the images. This communication is achieved through the use of a systematic mapping between graphic marks and data values in the creation of the visualization

In [ ]:
plt.figure(figsize=(25,20))
sns.factorplot(data=df_train,x='netgain',y='ratings',hue='genre')

In [ ]:
plt.figure(figsize=(15,10))
sns.catplot(x='realtionship_status',y='ratings',data=df_train,hue='netgain',height=5,aspect=3,kind='box')
plt.title('boxplot')

In [ ]:
plt.figure(figsize=(15,10))
sns.relplot(x='ratings', y='average_runtime(minutes_per_week)', data=df_train,
            kind='line', hue='netgain', col='expensive')

In [ ]:
plt.figure(figsize=(15,15))
sns.relplot(x='ratings', y='average_runtime(minutes_per_week)', data=df_train,
            kind='line')

In [ ]:
plt.figure(figsize=(10,10))
sns.catplot(x='realtionship_status',y='average_runtime(minutes_per_week)',data=df_train)

In [ ]:
plt.figure(figsize=(10,10))
sns.catplot(x='realtionship_status',y='average_runtime(minutes_per_week)',hue='netgain',data=df_train)

In [ ]:
plt.figure(figsize=(10,4))
sns.countplot(x='industry',hue='netgain',data=df_train,order=df_train['industry'].value_counts().sort_values().index);

In [ ]:
plt.figure(figsize=(10,4))
sns.countplot(x='targeted_sex',data=df_train,order=df_train['targeted_sex'].value_counts().sort_values().index,hue=df_train.netgain);

In [ ]:
plt.figure(figsize=(10,4))
sns.countplot(x='genre',hue='netgain',data=df_train,order=df_train['genre'].value_counts().sort_values().index);

In [ ]:
plt.figure(figsize=(15,7))
sns.countplot(x='expensive',hue='genre',data=df_train,order=df_train['expensive'].value_counts().sort_values().index);

In [ ]:
plt.figure(figsize=(11,5))
sns.countplot(x='genre',hue='netgain', data=df_train,palette="Set1")
plt.xlabel('Genre')
plt.ylabel('Count')
plt.title('Genre Wise Netgain')
plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = (10, 5)
sns.violinplot(df_train['airtime'], df_train['netgain'], palette = 'rainbow')
plt.title('Airtime vs Netgain Score', fontsize = 20)
plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = (10, 5)
sns.violinplot(df_train['expensive'], df_train['netgain'], palette = 'rainbow')
plt.title('expensive vs Netgain Score', fontsize = 20)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.distplot(df_train['average_runtime(minutes_per_week)'])
plt.show()

In [ ]:
sns.countplot(df_train.airtime,hue=df_train.netgain)

In [ ]:
def scatterplot(x_data, y_data, x_label="", y_label="", title="", color = "r", yscale_log=False):

    # Create the plot object
    _, ax = plt.subplots()

    # Plot the data, set the size (s), color and transparency (alpha)
    # of the points
    ax.scatter(x_data, y_data, s = 10, color = color, alpha = 0.75)

    if yscale_log == True:
        ax.set_yscale('log')

    # Label the axes and provide a title
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)

In [ ]:
scatterplot(df_train.genre,df_train.ratings,x_label="genre",y_label='airtime',color='blue',yscale_log=False)

In [ ]:
fig, ax = plt.subplots()
ax.scatter(x = df_train['ratings'], y = df_train['airlocation'])
plt.ylabel('Airlocation')
plt.xlabel('Ratings')
plt.show()

In [ ]:
# Plotting
sns.catplot(x='airtime', y='average_runtime(minutes_per_week)', data=df_train, kind='boxen', aspect=2)
plt.title('Boxen Plot', weight='bold', fontsize=16)
plt.show()

In [ ]:
sns.factorplot(data=df_train,x='industry',y='average_runtime(minutes_per_week)')
plt.title('Factor Plot', weight='bold', fontsize=16)
plt.show()

In [ ]:
# Plotting
sns.catplot(x='expensive', y='ratings', data=df_train, kind='boxen', aspect=2)
plt.title('Boxen Plot', weight='bold', fontsize=16)
plt.show()

In [ ]:
sns.set()
cols = ['realtionship_status','industry','genre','targeted_sex','average_runtime(minutes_per_week)','airtime','airlocation','ratings','expensive','money_back_guarantee','netgain']
sns.pairplot(df_train[cols], size = 2.5)
plt.show()

# One_hot Encoding
One hot encoding is a process by which categorical variables are converted into a form that could be provided to ML algorithms 
to do a better job in prediction.
The encoding is done using pandas.get_dummies

In [ ]:
encoded = pd.get_dummies(df_train)

In [ ]:
encoded.head()

In [ ]:
encoded.columns

In [ ]:
dependent_all=encoded['netgain']

In [ ]:
independent_all=encoded.drop(['id','netgain'],axis=1)

## Train And Test Split 

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(independent_all,dependent_all,test_size=0.3,random_state=100)

# LogisticRegression 

Logistic regression is a statistical model that in its basic form uses a logistic function to model a binary dependent variable, although many more complex extensions exist. In regression analysis, logistic regression (or logit regression) is estimating the parameters of a logistic model (a form of binary regression). Mathematically, a binary logistic model has a dependent variable with two possible values, such as pass/fail which is represented by an indicator variable, where the two values are labeled "0" and "1". In the logistic model, the log-odds (the logarithm of the odds) for the value labeled "1" is a linear combination of one or more independent variables ("predictors"); the independent variables can each be a binary variable (two classes, coded by an indicator variable) or a continuous variable (any real value). The corresponding probability of the value labeled "1" can vary between 0 (certainly the value "0") and 1 (certainly the value "1"),

In [ ]:
log =LogisticRegression()
log.fit(x_train,y_train)

In [ ]:
#model on train using all the independent values in df
log_prediction = log.predict(x_train)
log_score= accuracy_score(y_train,log_prediction)
print('Accuracy score on train set using Logistic Regression :',log_score)

# confusion matrix
A confusion matrix is a table that is often used to describe the performance of a classification model (or “classifier”) on a set of test data for which the true values are known. It allows the visualization of the performance of an algorithm.

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_train, log_prediction)

# AUC 
Compute Area Under the Curve (AUC) using the trapezoidal rule

In [ ]:
from sklearn import metrics
fpr, tpr, thresholds = metrics.roc_curve(y_train,log_prediction)
print("AUC on train using Logistic Regression :",metrics.auc(fpr, tpr))

# average precision recall score
Average precision (AP) summarizes such a plot as the weighted mean of precisions achieved at each threshold, with the increase in recall from the previous threshold used as the weight:

<math xmlns="http://www.w3.org/1998/Math/MathML">
  <mtext>AP</mtext>
  <mo>=</mo>
  <munder>
    <mo data-mjx-texclass="OP">&#x2211;</mo>
    <mi>n</mi>
  </munder>
  <mo stretchy="false">(</mo>
  <msub>
    <mi>R</mi>
    <mi>n</mi>
  </msub>
  <mo>&#x2212;</mo>
  <msub>
    <mi>R</mi>
    <mrow>
      <mi>n</mi>
      <mo>&#x2212;</mo>
      <mn>1</mn>
    </mrow>
  </msub>
  <mo stretchy="false">)</mo>
  <msub>
    <mi>P</mi>
    <mi>n</mi>
  </msub>
</math>

where <math xmlns="http://www.w3.org/1998/Math/MathML">
  <msub>
    <mi>P</mi>
    <mi>n</mi>
  </msub>
</math> and  <math xmlns="http://www.w3.org/1998/Math/MathML">
  <msub>
    <mi>R</mi>
    <mi>n</mi>
  </msub>
</math> are the precision and recall at the nth threshold. A pair  <math xmlns="http://www.w3.org/1998/Math/MathML">
  <mo stretchy="false">(</mo>
  <msub>
    <mi>R</mi>
    <mi>k</mi>
  </msub>
  <mo>,</mo>
  <msub>
    <mi>P</mi>
    <mi>k</mi>
  </msub>
  <mo stretchy="false">)</mo>
</math>   is referred to as an operating point.


In [ ]:
from sklearn.metrics import average_precision_score
average_precision = average_precision_score(y_train, log_prediction)

print('Average precision-recall score: {0:0.2f}'.format(
      average_precision))

# recall score

The recall is the ratio tp / (tp + fn) where tp is the number of true positives and fn the number of false negatives. The recall is intuitively the ability of the classifier to find all the positive samples.

The best value is 1 and the worst value is 0.

In [ ]:
from sklearn.metrics import recall_score
print('recall_score on train set :',recall_score(y_train, log_prediction))

   # F1 Score
Compute the F1 score, also known as balanced F-score or F-measure

The F1 score can be interpreted as a weighted average of the precision and recall, where an F1 score reaches its best value at 1 and worst score at 0. The relative contribution of precision and recall to the F1 score are equal. The formula for the F1 score is:

F1 = 2 * (precision * recall) / (precision + recall)

In [ ]:
from sklearn.metrics import f1_score
print('F1_sccore on train set :',f1_score(y_train, log_prediction))

In [ ]:
#model on train using all the independent values in df
log_prediction = log.predict(x_test)
log_score= accuracy_score(y_test,log_prediction)
print('accuracy score on test using Logisitic Regression :',log_score)

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, log_prediction)

In [ ]:
from sklearn import metrics
fpr, tpr, thresholds = metrics.roc_curve(y_test,log_prediction)
print("AUC on test using Logistic Regression :",metrics.auc(fpr, tpr))

In [ ]:
from sklearn.metrics import average_precision_score
average_precision = average_precision_score(y_test, log_prediction)

print('Average precision-recall score: {0:0.2f}'.format(
      average_precision))

In [ ]:
from sklearn.metrics import recall_score
print('recall_score on train set :',recall_score(y_test, log_prediction))

In [ ]:
from sklearn.metrics import f1_score
print('F1_sccore on train set :',f1_score(y_test, log_prediction))

In [ ]:
from sklearn.model_selection import cross_val_score
lr = LogisticRegression()
scores = cross_val_score(lr, x_train, y_train, cv=10, scoring = "accuracy")
print("Scores:", scores)
print("Mean:", scores.mean())
print("Standard Deviation:", scores.std())


<h3 >comparing metries [Logistic Regression ]</h3>
<table style="width:50%"> <thead> <tr> <th>Metries </th> <th> Train</th> <th> Test</th></tr> </thead> <tbody> <tr> <th scope='row'> Accuracy Score </th> <td>0.80402</td> <td>0.80330</td></tr> 
    <tr><th scope='row'>AUC </th> <td>0.6577</td> <td>0.6477</td></tr><tr> <th scope='row'>Average_precision_recall_score</th> <td>0.40</td> <td>0.38</td></tr> <tr> 
    <th scope='row'>recall_score </th> <td>0.3759</td> <td>0.35761
    </td></tr> <tr><th scope='row'>F1 Score </th> <td>0.4798</td> <td>0.4546</td></tr></table>

# XGBoost Algorithm

XGBoost is an optimized distributed gradient boosting library designed to be highly efficient, flexible and portable. It implements machine learning algorithms under the Gradient Boosting framework. XGBoost provides a parallel tree boosting (also known as GBDT, GBM) that solve many data science problems in a fast and accurate way. The same code runs on major distributed environment (Hadoop, SGE, MPI) and can solve problems beyond billions of examples.

In [ ]:
xgboost = xgb.XGBClassifier(max_depth=3,n_estimators=300,learning_rate=0.05)

In [ ]:
xgboost.fit(x_train,y_train)

In [ ]:
#XGBoost model on the train set
XGB_prediction = xgboost.predict(x_train)
XGB_score= accuracy_score(y_train,XGB_prediction)
print('accuracy score on train using XGBoost ',XGB_score)

In [ ]:
confusion_matrix(y_train, XGB_prediction)

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(y_train,XGB_prediction)
print("AUC on train using XGBoost :",metrics.auc(fpr, tpr))

In [ ]:
average_precision = average_precision_score(y_train, XGB_prediction)

print('Average precision-recall score: {0:0.2f}'.format(
      average_precision))

In [ ]:
print('recall_score on train set :',recall_score(y_train, XGB_prediction))

In [ ]:
print('F1_sccore on train set :',f1_score(y_train, XGB_prediction))

In [ ]:
#XGBoost model on the test
XGB_prediction = xgboost.predict(x_test)
XGB_score= accuracy_score(y_test,XGB_prediction)
print('accuracy score on test using XGBoost :',XGB_score)

In [ ]:
confusion_matrix(y_test, XGB_prediction)

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(y_test,XGB_prediction)
print("AUC on test using XGBoost :",metrics.auc(fpr, tpr))

In [ ]:
average_precision = average_precision_score(y_test, XGB_prediction)

print('Average precision-recall score: {0:0.2f}'.format(
      average_precision))

In [ ]:
print('recall_score on test set :',recall_score(y_test, XGB_prediction))

In [ ]:
print('F1_sccore on test set :',f1_score(y_test, XGB_prediction))

In [ ]:
xg = xgb.XGBClassifier()
scores = cross_val_score(xg, x_test, y_test, cv=10, scoring = "accuracy")
print("Scores:", scores)
print("Mean:", scores.mean())
print("Standard Deviation:", scores.std())


<h3 >comparing metries [XGBoost]</h3>
<table style="width:50%"> <thead> <tr> <th>Metries </th> <th> Train</th> <th> Test</th></tr> </thead> <tbody> <tr> <th scope='row'> Accuracy Score </th> <td>0.82141</td> <td>0.82070</td></tr> 
    <tr><th scope='row'>AUC </th> <td>0.69710</td> <td>0.686418</td></tr><tr> <th scope='row'>Average_precision_recall_score</th> <td>0.45</td> <td>0.43</td></tr> <tr> 
    <th scope='row'>recall_score </th> <td>0.45767</td> <td>0.43598
    </td></tr> <tr><th scope='row'>F1 Score </th> <td>0.55200</td> <td>0.53002</td></tr></table>

# RandomForestClassifier
Random forests or random decision forests are an ensemble learning method for classification, regression and other tasks that operate by constructing a multitude of decision trees at training time and outputting the class that is the mode of the classes (classification) or mean prediction (regression) of the individual trees.Random decision forests correct for decision trees' habit of overfitting to their training set.

In [ ]:
rfc2=RandomForestClassifier(n_estimators=300)
rfc2.fit(x_train,y_train)

In [ ]:
#model on train using all the independent values in df
rfc_prediction = rfc2.predict(x_train)
rfc_score= accuracy_score(y_train,rfc_prediction)
print('accuracy Score on train using RandomForest :',rfc_score)

In [ ]:
confusion_matrix(y_train, rfc_prediction)

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(y_train,rfc_prediction)
print("AUC on train using RandomForest :",metrics.auc(fpr, tpr))

In [ ]:
average_precision = average_precision_score(y_train, rfc_prediction)

print('Average precision-recall score: {0:0.2f}'.format(
      average_precision))

In [ ]:
print('recall_score on train set :',recall_score(y_train, rfc_prediction))

In [ ]:
print('F1_sccore on train set :',f1_score(y_train, rfc_prediction))

In [ ]:
#model on test using all the indpendent values in df
rfc_prediction = rfc2.predict(x_test)
rfc_score= accuracy_score(y_test,rfc_prediction)
print('accuracy score on test using RandomForest ',rfc_score)

In [ ]:
confusion_matrix(y_test, rfc_prediction)

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(y_test,rfc_prediction)
print("AUC on test using RandomForest :",metrics.auc(fpr, tpr))

In [ ]:
average_precision = average_precision_score(y_test, rfc_prediction)

print('Average precision-recall score: {0:0.2f}'.format(
      average_precision))

In [ ]:
print('recall_score on test set :',recall_score(y_test, rfc_prediction))

In [ ]:
print('F1_sccore on test set :',f1_score(y_test, rfc_prediction))

In [ ]:
lr = RandomForestClassifier(n_estimators=100)
scores = cross_val_score(lr, x_train, y_train, cv=10, scoring = "accuracy")
print("Scores:", scores)
print("Mean:", scores.mean())
print("Standard Deviation:", scores.std())


<h3 >comparing metries [RandomForest]</h3>
<table style="width:50%"> <thead> <tr> <th>Metries </th> <th> Train</th> <th> Test</th></tr> </thead> <tbody> <tr> <th scope='row'> Accuracy Score </th> <td>0.863097</td> <td>0.80752</td></tr> 
    <tr><th scope='row'>AUC </th> <td>0.76914</td> <td>0.676682</td></tr><tr> <th scope='row'>Average_precision_recall_score</th> <td>0.56</td> <td>0.40</td></tr> <tr> 
    <th scope='row'>recall_score </th> <td>0.58818</td> <td>0.43267
    </td></tr> <tr><th scope='row'>F1 Score </th> <td>0.673810</td> <td>0.51041</td></tr></table>

# DecisionTreeClassifer

A decision tree is a decision support tool that uses a tree-like model of decisions and their possible consequences, including chance event outcomes, resource costs, and utility. It is one way to display an algorithm that only contains conditional control statements.

Decision trees are commonly used in operations research, specifically in decision analysis, to help identify a strategy most likely to reach a goal, but are also a popular tool in machine learning.

In [ ]:
dec=DecisionTreeClassifier()

In [ ]:
dec.fit(x_train,y_train)

In [ ]:
#model on train using all the independent values in df
dec_prediction = dec.predict(x_train)
dec_score= accuracy_score(y_train,dec_prediction)
print('Accuracy score on train using Decision Tree :',dec_score)

In [ ]:
    print(confusion_matrix(y_train, dec_prediction))
    fpr, tpr, thresholds = metrics.roc_curve(y_train,dec_prediction)
    print("AUC on train using DecisionTree :",metrics.auc(fpr, tpr))
    average_precision = average_precision_score(y_train, dec_prediction)
    print('Average precision-recall score: {0:0.2f}'.format(average_precision))
    print('recall_score on train set :',recall_score(y_train, dec_prediction))
    print('F1_sccore on train set :',f1_score(y_train, dec_prediction))
    

In [ ]:
#model on test using all the independent values in df
dec_prediction = dec.predict(x_test)
dec_score= accuracy_score(y_test,dec_prediction)
print('Accuracy Score on tree using Decision Tree  :',dec_score)

In [ ]:
    print(confusion_matrix(y_test, dec_prediction))
    fpr, tpr, thresholds = metrics.roc_curve(y_test,dec_prediction)
    print("AUC on train using DecisionTree :",metrics.auc(fpr, tpr))
    average_precision = average_precision_score(y_test, dec_prediction)
    print('Average precision-recall score: {0:0.2f}'.format(average_precision))
    print('recall_score on test set :',recall_score(y_test, dec_prediction))
    print('F1_sccore on test set :',f1_score(y_test, dec_prediction))
    

In [ ]:
lr = DecisionTreeClassifier()
scores = cross_val_score(lr, x_train, y_train, cv=10, scoring = "accuracy")
print("Scores:", scores)
print("Mean:", scores.mean())
print("Standard Deviation:", scores.std())


<h3 >comparing metries [DecisionTree]</h3>
<table style="width:50%"> <thead> <tr> <th>Metries </th> <th> Train</th> <th> Test</th></tr> </thead> <tbody> <tr> <th scope='row'> Accuracy Score </th> <td>0.863097</td> <td>0.79779</td></tr> 
    <tr><th scope='row'>AUC </th> <td>0.76407</td> <td>0.66880</td></tr><tr> <th scope='row'>Average_precision_recall_score</th> <td>0.56</td> <td>0.38</td></tr> <tr> 
    <th scope='row'>recall_score </th> <td>0.573351</td> <td>0.428256
    </td></tr> <tr><th scope='row'>F1 Score </th> <td>0.668173</td> <td>0.49553</td></tr></table>

# ExtraTreeClassifier

Each Decision Tree in the Extra Trees Forest is constructed from the original training sample. Then, at each test node, Each tree is provided with a random sample of k features from the feature-set from which each decision tree must select the best feature to split the data based on some mathematical criteria (typically the Gini Index). This random sample of features leads to the creation of multiple de-correlated decision trees.

In [ ]:
etc=ExtraTreeClassifier()
etc.fit(x_train,y_train)

In [ ]:
#model on train using all the independent values in df
etc_prediction = etc.predict(x_train)
etc_score= accuracy_score(y_train,etc_prediction)
etc_score

In [ ]:
    print(confusion_matrix(y_train, etc_prediction))
    fpr, tpr, thresholds = metrics.roc_curve(y_train,etc_prediction)
    print("AUC on train using ExtraTree :",metrics.auc(fpr, tpr))
    average_precision = average_precision_score(y_train, etc_prediction)
    print('Average precision-recall score: {0:0.2f}'.format(average_precision))
    print('recall_score on train set :',recall_score(y_train, etc_prediction))
    print('F1_sccore on train set :',f1_score(y_train, etc_prediction))
    

In [ ]:
#model on test using all the independent values in df
etc_prediction = etc.predict(x_test)
etc_score= accuracy_score(y_test,etc_prediction)
etc_score

In [ ]:
    print(confusion_matrix(y_test, etc_prediction))
    fpr, tpr, thresholds = metrics.roc_curve(y_test,etc_prediction)
    print("AUC on train using ExtraTree :",metrics.auc(fpr, tpr))
    average_precision = average_precision_score(y_test, etc_prediction)
    print('Average precision-recall score: {0:0.2f}'.format(average_precision))
    print('recall_score on test set :',recall_score(y_test, dec_prediction))
    print('F1_sccore on test set :',f1_score(y_test, etc_prediction))
    

In [ ]:
lr = ExtraTreeClassifier()
scores = cross_val_score(lr, x_train, y_train, cv=10, scoring = "accuracy")
print("Scores:", scores)
print("Mean:", scores.mean())
print("Standard Deviation:", scores.std())


<h3 >comparing metries [ExtraTree]</h3>
<table style="width:50%"> <thead> <tr> <th>Metries </th> <th> Train</th> <th> Test</th></tr> </thead> <tbody> <tr> <th scope='row'> Accuracy Score </th> <td>0.863097</td> <td>0.794983</td></tr> 
    <tr><th scope='row'>AUC </th> <td>0.764074</td> <td>0.6592717</td></tr><tr> <th scope='row'>Average_precision_recall_score</th> <td>0.56</td> <td>0.37</td></tr> <tr> 
    <th scope='row'>recall_score </th> <td>0.573351</td> <td>0.428256
    </td></tr> <tr><th scope='row'>F1 Score </th> <td>0.668173</td> <td>0.478854</td></tr></table>

# AdaBoostClassifier

An AdaBoost [1] classifier is a meta-estimator that begins by fitting a classifier on the original dataset and then fits additional copies of the classifier on the same dataset but where the weights of incorrectly classified instances are adjusted such that subsequent classifiers focus more on difficult cases.

In [ ]:
ada =AdaBoostClassifier(n_estimators=100)

In [ ]:
ada.fit(x_train,y_train)

In [ ]:
#model on train using all the independent values in df
ada_prediction = ada.predict(x_train)
ada_score= accuracy_score(y_train,ada_prediction)
ada_score

In [ ]:
    print(confusion_matrix(y_train, ada_prediction))
    fpr, tpr, thresholds = metrics.roc_curve(y_train,ada_prediction)
    print("AUC on train using AdaBoost :",metrics.auc(fpr, tpr))
    average_precision = average_precision_score(y_train, ada_prediction)
    print('Average precision-recall score: {0:0.2f}'.format(average_precision))
    print('recall_score on train set :',recall_score(y_train, ada_prediction))
    print('F1_sccore on train set :',f1_score(y_train, ada_prediction))
    

In [ ]:
#model on test using all the independent values in df
ada_prediction = ada.predict(x_test)
ada_score= accuracy_score(y_test,ada_prediction)
print('accuracy score om test using AdaBoost :',ada_score)

In [ ]:
    print(confusion_matrix(y_test, ada_prediction))
    fpr, tpr, thresholds = metrics.roc_curve(y_test,ada_prediction)
    print("AUC on test using AdaBoost :",metrics.auc(fpr, tpr))
    average_precision = average_precision_score(y_test, ada_prediction)
    print('Average precision-recall score: {0:0.2f}'.format(average_precision))
    print('recall_score on test set :',recall_score(y_test, ada_prediction))
    print('F1_sccore on test set :',f1_score(y_test, ada_prediction))
    

In [ ]:
lr = AdaBoostClassifier(n_estimators=100)
scores = cross_val_score(lr, x_train, y_train, cv=10, scoring = "accuracy")
print("Scores:", scores)
print("Mean:", scores.mean())
print("Standard Deviation:", scores.std())


<h3 >comparing metries [AdaBoost]</h3>
<table style="width:50%"> <thead> <tr> <th>Metries </th> <th> Train</th> <th> Test</th></tr> </thead> <tbody> <tr> <th scope='row'> Accuracy Score </th> <td>0.818615</td> <td>0.816227</td></tr> 
    <tr><th scope='row'>AUC </th> <td>0.712104</td> <td>0.7008396</td></tr><tr> <th scope='row'>Average_precision_recall_score</th> <td>0.45</td> <td>0.43</td></tr> <tr> 
    <th scope='row'>recall_score </th> <td>0.5069587</td> <td>0.485651
    </td></tr> <tr><th scope='row'>F1 Score </th> <td>0.573345</td> <td>0.55068</td></tr></table>

# BaggingClassifier

A Bagging classifier is an ensemble meta-estimator that fits base classifiers each on random subsets of the original dataset and then aggregate their individual predictions (either by voting or by averaging) to form a final prediction. Such a meta-estimator can typically be used as a way to reduce the variance of a black-box estimator (e.g., a decision tree), by introducing randomization into its construction procedure and then making an ensemble out of it.

In [ ]:
bca =BaggingClassifier()
bca.fit(x_train,y_train)
#model on train using all the independent values in df
bca_prediction = bca.predict(x_train)
bca_score= accuracy_score(y_train,bca_prediction)
print('accuracy on train using BaggingClassifier :',bca_score)

In [ ]:
    print(confusion_matrix(y_train, bca_prediction))
    fpr, tpr, thresholds = metrics.roc_curve(y_train,bca_prediction)
    print("AUC on train using BaggingClassifier :",metrics.auc(fpr, tpr))
    average_precision = average_precision_score(y_train, bca_prediction)
    print('Average precision-recall score: {0:0.2f}'.format(average_precision))
    print('recall_score on train set :',recall_score(y_train, bca_prediction))
    print('F1_sccore on train set :',f1_score(y_train, bca_prediction))
    

In [ ]:
#model on test using all the independent values in df
bca_prediction = bca.predict(x_test)
bca_score= accuracy_score(y_test,bca_prediction)
print(bca_score)

In [ ]:
    print(confusion_matrix(y_test, bca_prediction))
    fpr, tpr, thresholds = metrics.roc_curve(y_test,bca_prediction)
    print("AUC on train using Bagging Classifier :",metrics.auc(fpr, tpr))
    average_precision = average_precision_score(y_test, bca_prediction)
    print('Average precision-recall score: {0:0.2f}'.format(average_precision))
    print('recall_score on test set :',recall_score(y_test, bca_prediction))
    print('F1_sccore on test set :',f1_score(y_test, bca_prediction))
    

In [ ]:
lr = BaggingClassifier(n_estimators=100)
scores = cross_val_score(lr, x_train, y_train, cv=10, scoring = "accuracy")
print("Scores:", scores)
print("Mean:", scores.mean())
print("Standard Deviation:", scores.std())

<h3 >comparing metries [Bagging Classifier]</h3>
<table style="width:50%"> <thead> <tr> <th>Metries </th> <th> Train</th> <th> Test</th></tr> </thead> <tbody> <tr> <th scope='row'> Accuracy Score </th> <td>0.859258</td> <td>0.806117</td></tr> 
    <tr><th scope='row'>AUC </th> <td>0.76973</td> <td>0.68462</td></tr><tr> <th scope='row'>Average_precision_recall_score</th> <td>0.55</td> <td>0.40</td></tr> <tr> 
    <th scope='row'>recall_score </th> <td>0.59730</td> <td>0.45805
    </td></tr> <tr><th scope='row'>F1 Score </th> <td>0.677110</td> <td>0.52283</td></tr></table>

# ExtraTreesClassifier

Extremely Randomized Trees Classifier(Extra Trees Classifier) is a type of ensemble learning technique which aggregates the results of multiple de-correlated decision trees collected in a “forest” to output it’s classification result. In concept, it is very similar to a Random Forest Classifier and only differs from it in the manner of construction of the decision trees in the forest.

In [ ]:
ettc=ExtraTreesClassifier()
ettc.fit(x_train,y_train)
#model on train using all the independent values in df
ettc_prediction = ettc.predict(x_train)
ettc_score= accuracy_score(y_train,ettc_prediction)
print('training accuracy using Extratressclassifier',ettc_score)

In [ ]:
    print(confusion_matrix(y_train, ettc_prediction))
    fpr, tpr, thresholds = metrics.roc_curve(y_train,ettc_prediction)
    print("AUC on train using ExtraTree :",metrics.auc(fpr, tpr))
    average_precision = average_precision_score(y_train, ettc_prediction)
    print('Average precision-recall score: {0:0.2f}'.format(average_precision))
    print('recall_score on train set :',recall_score(y_train, ettc_prediction))
    print('F1_sccore on train set :',f1_score(y_train, ettc_prediction))
    

In [ ]:
#model on test using all the independent values in df
ettc_prediction =ettc.predict(x_test)
ettc_score= accuracy_score(y_test,ettc_prediction)
print('testing accuracy using Extratressclassifier',ettc_score)

In [ ]:
    print(confusion_matrix(y_test, ettc_prediction))
    fpr, tpr, thresholds = metrics.roc_curve(y_test,ettc_prediction)
    print("AUC on test using Extratreesclassifier :",metrics.auc(fpr, tpr))
    average_precision = average_precision_score(y_test, ettc_prediction)
    print('Average precision-recall score: {0:0.2f}'.format(average_precision))
    print('recall_score on test set :',recall_score(y_test, ettc_prediction))
    print('F1_sccore on test set :',f1_score(y_test, ettc_prediction))
    

<h3 >comparing metries [ExtraTreesClassifier]</h3>
<table style="width:50%"> <thead> <tr> <th>Metries </th> <th> Train</th> <th> Test</th></tr> </thead> <tbody> <tr> <th scope='row'> Accuracy Score </th> <td>0.863097</td> <td>0.801254</td></tr> 
    <tr><th scope='row'>AUC </th> <td>0.764074</td> <td>0.663161</td></tr><tr> <th scope='row'>Average_precision_recall_score</th> <td>0.56</td> <td>0.38</td></tr> <tr> 
    <th scope='row'>recall_score </th> <td>0.573351</td> <td>0.405629
    </td></tr> <tr><th scope='row'>F1 Score </th> <td>0.6681733</td> <td>0.486271</td></tr></table>

# Neural Network(MLPclassifier)

In [ ]:
nn=MLPClassifier()
nn.fit(x_train,y_train)
nn_prediction = nn.predict(x_train)
nn_score= accuracy_score(y_train,nn_prediction)
print('accuracy score on train using MLPClassifier :',nn_score)
#model on test using all the independent values in df
nn_prediction =nn.predict(x_test)
nn_score= accuracy_score(y_test,nn_prediction)
print('accuracy score on test using MLPClassifier :',nn_score)

## feature Enginearing on test data 

In [ ]:
df_test.head()

In [ ]:
encoded = pd.get_dummies(df_test)

In [ ]:
encoded

In [ ]:
testing=encoded.drop(['id'],axis=1)

<h3 style='padding: 10px'>comparison table</h2><table border-style:solid; class='table table-striped'> <thead> <tr> <th>Algorithm Used</th> <th>Accuracy Score On Train</th> <th>Accuracy Score On Test</th></tr> </thead> <tbody> <tr> <th scope='row'>XGBoost Classifier </th> <td>0.82141290039491</td> <td>0.8207064243665216</td></tr> 
    <tr> <th scope='row'>Random Forest Classifier</th> <td>0.8588196577446249</td> <td>0.8050934220629639</td></tr> <tr> 
    <th scope='row'>Logisitic Regresion</th> <td>0.8040258885476086</td> <td>0.8033017660609163
    </td></tr> <tr><th scope='row'>Decision Tree Classifier</th> <td>0.8630978499341817</td> <td>0.7999744049142564</td></tr>
    <tr><th scope='row'>Extra tree classifier</th><td>0.8630978499341817</td><td>0.8001023803429741</td></tr>
    <tr><th scope='row'>ADA boost classifier</th><td>0.8180122860903906</td><td>0.8162272843614026</td></tr>
    <tr><th scope='row'>Bagging classifier</th><td>0.8593132953049584</td><td>0.8052213974916816</td></tr>
    <tr><th scope='row'>ExtraTreesclassifier</th><td>0.8630978499341817</td><td>0.8004863066291272</td></tr>
    
        <tr><th scope='row'>NeuralNetwork(MLPClassifier)</th><td>0.8099495392716104</td><td>0.8033017660609163</td></tr>
    </tbody> </table>

#  Best Algorithm

From the above table we get to known that XGBoost algorithm gives us the best accuracy score and produces less delta value 

XGBoost model is to Predict the targeted variable on the test variable 

In [ ]:
prediction = xgboost.predict(testing)

In [ ]:
submission=pd.DataFrame({"id":df_test.id,'prediction': prediction})

In [ ]:
# saving the dataframe 
submission.to_csv('sample_submission.csv')

In [ ]:
import pandas as pd
%matplotlib inline
#do code to support model
#"data" is the X dataframe and model is the SKlearn object

feats = {} # a dict to hold feature_name: feature_importance
for feature, importance in zip(x_train.columns, rfc2.feature_importances_):
    feats[feature] = importance #add the name/value pair 

importances = pd.DataFrame.from_dict(feats, orient='index').rename(columns={0: 'Gini-importance'})
#plt.figure(figsize=(15,7))
importances.sort_values(by='Gini-importance').plot(kind='bar', rot=45,figsize=(15,7))

In [ ]:
importances = rfc2.feature_importances_
std = np.std([tree.feature_importances_ for tree in rfc2.estimators_],
             axis=0)
indices = np.argsort(importances)[::-1]

# Print the feature ranking
print("Feature ranking:")

for f in range(x_train.shape[1]):
    print("%d. feature %d (%f)" % (f + 1, indices[f], importances[indices[f]]))

# Plot the feature importances of the forest
#plt.figure()
plt.figure(figsize=(15,7))
plt.title("Feature importances")
plt.bar(range(x_train.shape[1]), importances[indices],
       color="r", yerr=std[indices], align="center")
plt.xticks(range(x_train.shape[1]), indices)
plt.xlim([-1, x_train.shape[1]])
plt.show()
